# Ataque ALGEN sentence-level sobre embeddings DPED (SST-2)

Sequência de execução: (1) re-export DPED com `token_freq.npy`, (2) smoke test do atacante, (3) grade de ataque DPED e baseline `bert_pretrained`, (4) cópia dos resultados para o Drive.

Plano: [plano-ataque-algen-dped.md](../docs/analises/plano-ataque-algen-dped.md). Implementação: [ataque-algen-sentence-level.md](../docs/implementação/ataque-algen-sentence-level.md).

Antes de executar: ajuste `ALGEN_CKPT` para a pasta do checkpoint do decodificador ALGEN no Drive. Ele precisa conter `training_args_and_best_models.json`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!git -C /content/DPED-Sistemas-Hibridos pull || git clone https://github.com/otavio-hbf/DPED-Sistemas-Hibridos.git /content/DPED-Sistemas-Hibridos

In [ ]:
%cd /content/DPED-Sistemas-Hibridos
!pip install -q -r requirements.txt

In [ ]:
import os
import subprocess
import sys
import shutil
from datetime import datetime

REPO = '/content/DPED-Sistemas-Hibridos'
os.chdir(REPO)

# Ajustar para a pasta do checkpoint do decodificador ALGEN no Drive
ALGEN_CKPT = '/content/drive/MyDrive/SIH/algen/checkpoint'
DRIVE_RUNS = '/content/drive/MyDrive/SIH/privEmbeddings'
RUN_DATE = datetime.now().strftime('%Y-%m-%d')

EPSILONS = ['8.0', '4.0', '2.0', '1.0']
TRAIN_SIZES = [1, 10, 30, 100, 300, 500, 1000]
# Ridge fixo até a decisão P6 (escolha por validação ou fixo)
RIDGE_VALUES = [1.0]
ATTACKER = 'algen/SIH-ALGEN-main/src/attacker_dped_sentence.py'

def export_dir_for(eps):
    tag = eps.replace('.', 'p')
    return f'results/embeddings/glue_sst2_dp-distill_eps{tag}/dped_student_pre_downstream'

EXPORT_EPS8 = export_dir_for('8.0')
print('ALGEN_CKPT exists:', os.path.isdir(ALGEN_CKPT))

## Etapa 1: re-export DPED com `token_freq.npy`

Os comandos são os mesmos de `dped_medium.ipynb`. O re-export grava `token_freq.npy`, necessário ao atacante na opção (c) da máscara de X.

In [ ]:
!python main.py --dataset glue --task sst2 --method dp-distill --epsilon 8.0 --num_epochs 5 --batch_size 16 --max_train_samples 16384 --max_eval_samples 4096 --max_test_samples 1024 --max_public_samples 1024 --subset_strategy random --subset_seed 42

In [ ]:
!python main.py --dataset glue --task sst2 --method dp-distill --epsilon 4.0 --num_epochs 5 --batch_size 16 --max_train_samples 16384 --max_eval_samples 4096 --max_test_samples 1024 --max_public_samples 1024 --subset_strategy random --subset_seed 42

In [ ]:
!python main.py --dataset glue --task sst2 --method dp-distill --epsilon 2.0 --num_epochs 5 --batch_size 16 --max_train_samples 16384 --max_eval_samples 4096 --max_test_samples 1024 --max_public_samples 1024 --subset_strategy random --subset_seed 42

In [ ]:
!python main.py --dataset glue --task sst2 --method dp-distill --epsilon 1.0 --num_epochs 5 --batch_size 16 --max_train_samples 16384 --max_eval_samples 4096 --max_test_samples 1024 --max_public_samples 1024 --subset_strategy random --subset_seed 42

In [ ]:
!ls {EXPORT_EPS8}

## Etapa 2: smoke test do atacante

Poucos exemplos, para validar carregamento do checkpoint, export e escrita dos CSVs antes da grade completa.

In [ ]:
!python {ATTACKER} --checkpoint_path {ALGEN_CKPT} --export_dir {EXPORT_EPS8} --source dped --epsilon 8.0 --train_samples 10 --test_samples 10 --ridge 1.0 --output_dir results/attack/smoke

## Etapa 3: grade de ataque

- DPED: uma execução por ε × `train_samples` × ridge.
- Baseline `bert_pretrained`: lê as linhas do BERT para os mesmos tokens. Usa o export de ε=8.0, porque o conjunto de tokens e as frequências vêm desse export. O baseline não depende de ε.

Os resultados vão para `results/attack/`. Cada execução gera `sentence_results.csv` e `summary.json`.

In [ ]:
def run_attacker(args):
    cmd = [sys.executable, ATTACKER, '--checkpoint_path', ALGEN_CKPT] + [str(a) for a in args]
    print(' '.join(cmd))
    subprocess.run(cmd, check=True)

for eps in EPSILONS:
    tag = eps.replace('.', 'p')
    for n in TRAIN_SIZES:
        for r in RIDGE_VALUES:
            run_attacker(['--export_dir', export_dir_for(eps), '--source', 'dped', '--epsilon', eps,
                          '--train_samples', n, '--test_samples', 200, '--ridge', r,
                          '--output_dir', f'results/attack/dped_eps{tag}/n{n}_r{r}'])

for n in TRAIN_SIZES:
    for r in RIDGE_VALUES:
        run_attacker(['--export_dir', EXPORT_EPS8, '--source', 'bert_pretrained',
                      '--train_samples', n, '--test_samples', 200, '--ridge', r,
                      '--output_dir', f'results/attack/bert_pretrained/n{n}_r{r}'])

## Etapa 4: salvar resultados no Drive

In [ ]:
dst = os.path.join(DRIVE_RUNS, RUN_DATE, 'results_attack')
shutil.copytree(os.path.join(REPO, 'results', 'attack'), dst, dirs_exist_ok=True)
print('Resultados copiados para:', dst)